In [0]:
# Add widgets for parameters
dbutils.widgets.text("source_path", "/Volumes/workspace/ecommerce/ecommerce_data/delta/ng_nov_events")
dbutils.widgets.dropdown("layer", "bronze", ["bronze","silver","gold"])


In [0]:
# Use parameters
source = dbutils.widgets.get("source_path")
layer = dbutils.widgets.get("layer")

In [0]:
print(source)
print(layer)

/Volumes/workspace/ecommerce/ecommerce_data/delta/ng_nov_events/
silver


In [0]:
from pyspark.sql import functions as F

def run_layer(layer_name):
    # BRONZE: Raw ingestion
    if layer_name == "bronze":
        path = source
        raw = spark.read.format("delta").load(path)
        raw.withColumn("ingestion_ts", F.current_timestamp()) \
            .write.format("delta").mode("overwrite")\
            .save("/Volumes/workspace/ecommerce/ecommerce_data/delta/layers/bronze/events")
    # SILVER: Cleaned data
    elif layer_name == "silver":
        bronze = spark.read.format("delta")\
        .load("/Volumes/workspace/ecommerce/ecommerce_data/delta/layers/bronze/events")
        silver = bronze.filter(F.col("price") > 0) \
            .filter(F.col("price") < 10000) \
            .dropDuplicates(["user_session", "event_time"]) \
            .withColumn("event_date", F.to_date("event_time")) \
            .withColumn("price_tier",
                        F.when(F.col("price") < 10, "budget").when(F.col("price") < 50, "mid")
                        .otherwise("premium"))
        silver.write.format("delta")\
            .mode("overwrite").save("/Volumes/workspace/ecommerce/ecommerce_data/delta/layers/silver/events")
     # GOLD: Aggregated data
    elif layer_name == "gold":
        silver = spark.read.format("delta")\
            .load("/Volumes/workspace/ecommerce/ecommerce_data/delta/layers/silver/events")
        product_perf = silver.groupBy("product_id") \
                .agg(F.countDistinct(F.when(F.col("event_type")=="view", "user_id")).alias("views"),
                     F.countDistinct(F.when(F.col("event_type")=="purchase", "user_id")).alias("purchases"),
                     F.sum(F.when(F.col("event_type")=="purchase", F.col("price"))).alias("revenue")
                     ).withColumn("conversion_rate", F.try_divide(F.col("purchases"),F.col("views"))*100)
        product_perf.write.format("delta").mode("overwrite")\
            .save("/Volumes/workspace/ecommerce/ecommerce_data/delta/layers/gold/events")

In [0]:
run_layer(layer)